

The exported `vehicle_sensor_model` and `disturbance_ekf_sensor_correct` extend
the masked update to depth, RPY, DVL velocity, body gyro and specific force.
Specific force includes gravity, body-coordinate transport and a configured IMU
lever arm; it is not a numerically differentiated DVL observation. The original
seven-measurement exports remain available, and masking all six raw IMU channels
is mathematically equivalent to the legacy correction. Inputs are nominal model
parameters and measured sensor data, never disturbance ground truth. Process
intensities express a tracking/noise assumption; they are not sensor calibration.


# Nominal-dynamics disturbance EKF

State: `[pose_ned(6), velocity_body(6), disturbance_body(6)]`.
Acceleration is a derived output, not an independent state.

The implementation lives in `diffUV/estimation.py`; this notebook uses the same
CasADi → generated C → GCC shared-library pipeline. See `diffUV/estimation.py`
for input conventions, covariance semantics and limitations. Running this notebook does not install or replace simulator libraries.

`disturbance_ekf_predict_covariance` accepts continuous process intensity and
propagates noise through the same nominal dynamics with three-point quadrature.
It retains within-step motion/wrench cross-covariance. Subdivide intervals above
20 ms and check convergence for the intended dynamics. The original exports
continue to accept discrete process covariance; their ABI is unchanged.


In [ ]:
# Deployment export is centralized in code_generation.ipynb: export_vehicle(output, cuda=False).
# This notebook remains an EKF model/validation example; do not export a separate model version here.


## Use the canonical exporter
Run `code_generation.ipynb` to define `export_vehicle`, then call it with a new output directory. This notebook constructs the same model functions for analysis without writing artifacts.


In [ ]:
from diffUV.estimation import disturbance_ekf_functions, vehicle_sensor_model
prediction, correction = disturbance_ekf_functions()
sensors = vehicle_sensor_model()
print(prediction)
print(correction)
print(sensors)


## Validate before integration

Validate the generated functions against the simulator observer tests before installation.

C++ integration is a separate step. Supply canonical-frame sensor measurements,
preceding-interval actuator-effort estimates, fixed nominal parameters, positive
sample periods and valid covariance matrices. Do not supply actual plant
parameters or injected disturbance truth. The existing installed plant step
must be checked against this source's discretization before comparison trials.


## Experimental load-reference model

The optional `disturbance_process_model()` in the source is an experimental prior, not part of the deployed EKF export. Noise tests exposed tracking/noise trade-offs; it is not enabled in the simulator. The current ideal-sensor simulator correction belongs in its measurement/noise configuration, not in a tool-specific wrench prior.
